# snnTorch Tutorial 6 baseline — notebook runner

This notebook runs the **real** training + validation + test-set evaluation +
energy measurement path, not a toy demo. It calls
`neuron_baseline.runner.run_training(cfg)` — the same function
`python -m neuron_baseline train` uses — so notebook results and CLI results
cannot drift.

Artifacts land under `../runs/<timestamp>/` exactly like a CLI run:
`config.json`, `checkpoints/{best,last}.pt`, `results.json`,
`plots/{loss,accuracy}.png`.

See `../README.md` for setup, the model spec, and the measurement caveats.

In [ ]:
import sys, pathlib
# Make the sibling package importable when running from `notebooks/`.
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import torch
print('torch', torch.__version__, 'cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## 1. Build the config

`Config()` gives the tutorial-faithful defaults. Edit any field here — the
resolved config is what gets saved into `runs/<ts>/config.json`. Common
notebook overrides: `epochs`, `seed`, `batch_size`.

In [ ]:
from neuron_baseline.config import Config
cfg = Config()
# Notebook overrides — uncomment / edit as needed:
# cfg.epochs = 5
# cfg.seed = 42
print(cfg.to_json())

## 2. Sanity-check the model

One forward pass on random input at a short `T` — just proves the network
wires up before spending an epoch on it.

In [ ]:
from neuron_baseline.model import build_net, forward_pass

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = build_net(cfg.beta, cfg.surrogate_slope, cfg.threshold, cfg.reset_mechanism).to(device)
x = torch.rand(4, 1, 28, 28, device=device)
spk, mem = forward_pass(net, num_steps=8, data=x)
print('spk', tuple(spk.shape), 'mem', tuple(mem.shape))
print('mean spikes per (step, neuron):', float(spk.mean()))
del net, spk, mem, x  # runner rebuilds a fresh net

## 3. Peek at the data loaders

`build_loaders` carves a fixed 5000-example validation split off the shuffled
MNIST training set. The runner also calls this internally — this cell is
just to confirm the sizes look right.

In [ ]:
from neuron_baseline.data import build_loaders
loaders = build_loaders(cfg.data_dir, cfg.batch_size, cfg.eval_batch_size,
                        cfg.val_size, cfg.seed)
print('train batches', len(loaders['train']))
print('val examples ', len(loaders['val'].dataset))
print('test examples', len(loaders['test'].dataset))

## 4. Train + energy-measured test-set eval

One call runs everything the CLI does:

1. Seed, save `config.json`, fingerprint hardware/software.
2. Train for `cfg.epochs` epochs; per-epoch validation; save `best.pt` on best
   validation accuracy, `last.pt` at the end.
3. Reload `best.pt`, warm up, then measure inference energy on the reserved
   MNIST test set.
4. Write `results.json` and render loss/accuracy PNGs.

The current energy scope is **GPU-only** (NVML). Training and per-epoch
validation are inside one window; inference is a separate window.

In [ ]:
from neuron_baseline.runner import run_training

out = run_training(cfg)   # verbose=True prints [cfg]/[data]/[energy]/[epoch]/[done]
run_dir = out['run_dir']
results = out['results']

print()
print('run_dir      :', run_dir)
print('best_val_acc :', results['best_val_accuracy'])
print('test         :', results['test'])
print('wall_time_s  :', results['wall_time_s'])
print('train energy :', results['energy']['train'])
print('infer energy :', results['energy']['inference'])
print('J / image    :', results['energy']['inference_per_image_j'])

## 5. Independently re-evaluate `best.pt` on the test set

Same function `cmd eval` uses. Handy for double-checking, or for testing
a checkpoint from a different run:

```
ckpt_path = pathlib.Path('../runs/<ts>/checkpoints/best.pt')
```

In [ ]:
from neuron_baseline.eval import evaluate

ckpt_path = run_dir / 'checkpoints' / 'best.pt'
ckpt = torch.load(str(ckpt_path), map_location=device, weights_only=False)
eval_net = build_net(cfg.beta, cfg.surrogate_slope, cfg.threshold, cfg.reset_mechanism).to(device)
eval_net.load_state_dict(ckpt['model_state'])

test_metrics = evaluate(eval_net, loaders['test'], cfg.num_steps, device)
print('re-eval on best.pt:', test_metrics)

## 6. Inline the plots and inspect `results.json`

In [ ]:
import json
from IPython.display import Image, display

results_on_disk = json.loads((run_dir / 'results.json').read_text(encoding='utf-8'))
print('keys:', list(results_on_disk.keys()))
print('epoch_val_acc:', results_on_disk['history']['epoch_val_acc'])
print('epoch_time_s :', results_on_disk['history']['epoch_time_s'])

for name in ('loss.png', 'accuracy.png'):
    p = run_dir / 'plots' / name
    if p.exists():
        print(p)
        display(Image(filename=str(p)))